In [1]:
#Importation des fonctions(bibliothéque)
from pyspark.sql.functions import col, round, dayofmonth, month,quarter,year,when,regexp_replace,substring

StatementMeta(, f868766d-bafd-4348-9f9a-cd6da7629f7b, 4, Finished, Available, Finished, False)

In [2]:
# Recuperer le chemin de la table dans le lakehouse bronze
Chemin_Table_Bronze="abfss://WindPowerAnalytic_Dev@onelake.dfs.fabric.microsoft.com/LH_Wind_Power_Bronze.Lakehouse/Tables/dbo/wind_power"
# Charger la table dans le lakehouse bronze
df=spark.read.format("delta").load(Chemin_Table_Bronze)

StatementMeta(, f868766d-bafd-4348-9f9a-cd6da7629f7b, 5, Finished, Available, Finished, False)

In [ ]:
#Clean and enrich data
df_transformation = (
    df
    .withColumn("wind_speed", round(col("wind_speed"), 2))
    .withColumn("energy_produced", round(col("energy_produced"), 2))
    .withColumn("Day", dayofmonth(col("date")))
    .withColumn("Month", month(col("date")))
    .withColumn("Quarter", quarter(col("date")))
    .withColumn("Year", year(col("date")))
    .withColumn("time", regexp_replace(col("time"), "-", ":"))
    .withColumn("Heure", substring(col("time"), 1, 2).cast("int"))
    .withColumn("Minute", substring(col("time"), 4, 2).cast("int"))
    .withColumn("Seconde", substring(col("time"), 7, 2).cast("int"))
    .withColumn(
        "PeriodeDeTemps",
        when(col("Heure").between(5,11), "Matin")
        .when(col("Heure").between(12,16), "Après-midi")
        .when(col("Heure").between(17,21), "Soir")
        .otherwise("Nuit")
    )
)

StatementMeta(, f868766d-bafd-4348-9f9a-cd6da7629f7b, 6, Finished, Available, Finished, False)

In [ ]:
# Recuperer le chemin de la table dans le lakehouse silver
Chemin_Table_Silver="abfss://WindPowerAnalytic_Dev@onelake.dfs.fabric.microsoft.com/LH_Wind_Power_Silver.Lakehouse/Tables/dbo/wind_power"
#Enregistrer les données transformées dans le lakehouse silver
df_transformation.write.format("delta").mode("overwrite").save(Chemin_Table_Silver)

StatementMeta(, f868766d-bafd-4348-9f9a-cd6da7629f7b, 8, Finished, Available, Finished, False)